# Day 03 — Stability analysis (CV, bootstrap, leakage)

**Slides (tải về):** [`day03_slides.pptx`](_static/slides/day03_slides.pptx)

Mục tiêu:
- Chạy 5-fold cross validation AUC.
- Bootstrap để lấy khoảng tin cậy cho AUC.
- Hiểu leakage và cách tránh.

Gợi ý: dùng feature set tốt nhất từ Day 02.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
plt.rcParams['figure.figsize'] = (6,4)


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

# Nếu chạy trên Colab và không thấy file, notebook sẽ tải demo CSV từ GitHub.
GITHUB_USER = "YOUR_GITHUB_USERNAME"
REPO_NAME = "YOUR_REPO_NAME"
BRANCH = "main"

def download_from_github(rel_path: str) -> Path:
    import urllib.request
    url = f"https://raw.githubusercontent.com/{GITHUB_USER}/{REPO_NAME}/{BRANCH}/{rel_path}"
    out_path = Path(rel_path).name
    print("Tải file:", url)
    urllib.request.urlretrieve(url, out_path)
    return Path(out_path)

def load_csv(rel_path: str, fallback_upload: bool = True) -> pd.DataFrame:
    candidates = [
        Path(rel_path),
        Path("../")/rel_path,
        Path("../../")/rel_path,
        Path("data")/Path(rel_path).name,
        Path("_static/data")/Path(rel_path).name,
        Path("../data")/Path(rel_path).name,
    ]
    for p in candidates:
        if p.exists():
            return pd.read_csv(p)

    # Thử tải từ GitHub
    try:
        p = download_from_github(rel_path)
        return pd.read_csv(p)
    except Exception as e:
        print("Không tải được từ GitHub:", e)

    # Cuối cùng: upload thủ công
    if fallback_upload:
        try:
            from google.colab import files  # type: ignore
            uploaded = files.upload()
            if len(uploaded) > 0:
                up_path = Path(list(uploaded.keys())[0])
                return pd.read_csv(up_path)
        except Exception:
            pass

    raise FileNotFoundError("Không tìm thấy CSV. Cần clone repo hoặc upload file.")

df = load_csv("data/nsclc_egfr_radiomics_simplified.csv")
df.head()


In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

y = df["egfr_mutation"].astype(int)

# chọn 1 feature set để minh hoạ (có thể đổi intra/ring1/ring3/ring5/clinical)
feature_set = "ring1"
cols = [c for c in df.columns if c.startswith(feature_set + "_")] if feature_set != "clinical" else ["age","sex","smoking_status","histology","stage","tumor_size_mm","tumor_volume_cm3"]
X = df[cols].copy()

cat_cols = [c for c in X.columns if X[c].dtype == "object"]
num_cols = [c for c in X.columns if c not in cat_cols]

pre = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
    ],
    remainder="drop"
)

model = LogisticRegression(max_iter=2000)
pipe = Pipeline(steps=[("pre", pre), ("model", model)])

print("Feature set:", feature_set, "| n_features:", X.shape[1])


## 1) Cross validation AUC

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(pipe, X, y, cv=cv, scoring="roc_auc")
scores, scores.mean(), scores.std(ddof=1)


## 2) Bootstrap AUC (train test lặp)
Bootstrap dưới đây là mô phỏng đơn giản: mỗi lần lấy mẫu lại dữ liệu, chia train test, train và tính AUC.

Mục tiêu: nhìn phân phối AUC và ước lượng CI.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

def bootstrap_auc(X, y, n_iter=200, test_size=0.25, seed=42):
    rng = np.random.default_rng(seed)
    aucs = []
    n = len(y)
    for _ in range(n_iter):
        idx = rng.integers(0, n, size=n)  # resample with replacement
        Xb = X.iloc[idx].reset_index(drop=True)
        yb = y.iloc[idx].reset_index(drop=True)
        X_train, X_test, y_train, y_test = train_test_split(
            Xb, yb, test_size=test_size, stratify=yb, random_state=rng.integers(0, 10**9)
        )
        pipe.fit(X_train, y_train)
        proba = pipe.predict_proba(X_test)[:,1]
        aucs.append(roc_auc_score(y_test, proba))
    return np.array(aucs)

aucs = bootstrap_auc(X, y, n_iter=200)
np.mean(aucs), np.std(aucs, ddof=1), np.quantile(aucs, [0.025, 0.5, 0.975])


In [ ]:
plt.hist(aucs, bins=20)
plt.title("Bootstrap AUC distribution")
plt.xlabel("AUC")
plt.ylabel("count")
plt.show()


## 3) Leakage (nhắc nhanh)
Leakage là khi thông tin của test bị dùng trong train.

Ví dụ hay gặp:
- Scale toàn bộ dữ liệu trước khi chia train test.
- Chọn feature dựa trên toàn bộ dữ liệu rồi mới chia.

Cách tránh:
- Dùng Pipeline (scaler và encoder nằm trong pipeline).
- Nếu có tuning hoặc chọn feature, đặt vào trong cross validation.

Buổi này không yêu cầu viết code leakage, chỉ cần hiểu và nói được.

## Bài tập cuối buổi
1) Mean ± std AUC từ CV là bao nhiêu?
2) Bootstrap CI 95% của AUC là bao nhiêu?
3) Nếu CI rộng, cần làm gì để kết quả chắc hơn?
4) Nêu 1 ví dụ leakage và 1 cách tránh.